# Preprocessing

Imports and system path

In [1]:
import multiprocessing as mp
import os
import sys
from functools import partial
from itertools import chain

import numpy as np
import pandas as pd
import trimesh
from numba import njit
from scipy.spatial import KDTree
from tqdm import tqdm

# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


Processing `npt-HK4.gro` file into pandas dataframe

In [2]:
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
path = 'data/npt-HK4.gro'
file = os.path.join(project_root, path)

# Extracts data from .gro file into multi-index DataFrame (unsorted)
df_gro, title, num_atoms, box_dimensions = gp.read_gro(file, multiply=10, positions=True, velocities=False) # convert nm to Å  
    
# Checking
# df_gro
# molecules # display 

# Generating Molecule Meshes

Create `mol_meshes` dictionary, which contains 1501 individual molecule mesh. (i.e. `{1: mesh1, 2: mesh2, ... 1501: mesh1501}`)

In [3]:
from utils.generate_mol_meshes import molecules_to_meshes

mol_meshes = molecules_to_meshes(df_gro, box_dimensions, num_processes=None, context='fork')
print(len(mol_meshes))        # 1501
print(mol_meshes[1])          # trimesh.Trimesh object

Processing 1501 molecules with 8 logical cores: 100%|██████████| 1501/1501 [00:54<00:00, 27.79it/s]


1501
<trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>


# Export meshes

Users have two options to save `.npz` file or `.ply` file.
- `.npz` is much smaller ~400Mb, but exports longer $~50\,\mathrm{s}$ and import $~20\,\mathrm{s}$ 
- `.ply` file is larger ~1.00GB, but exports faster $~20\,\mathrm{s}$ and import $~10\,\mathrm{s}$

In [4]:
from utils.generate_mol_meshes import export_meshes

export_meshes(mol_meshes, path=path, export_format='ply', num_processes=None, context='fork')

Exporting 1501 meshes with 8 logical cores: 100%|██████████| 1501/1501 [00:09<00:00, 164.60it/s]


Successfully saved all meshes into directory: npt-HK4_meshes
File size: 1087.88 MB


# Import meshes (TODO)

In [3]:
import numpy as np
import trimesh

def import_meshes(file):
    """
    Extracts all mesh components from .npz, then reconstructs trimesh objects,
    and stores them in a dictionary using 1-based indexing.
    
    This method assumes that each mesh has 3 separate data types, vertices, faces, and colors.

    Args:
        file_path (str): The path to the NPZ file. (e.g. 'npt-HK4_meshes.npz')

    Returns:
        dict: A dictionary of {mesh_id: trimesh.Trimesh object}.
    """
   # Note: If load a .npz file, it becomes it's own class.
   # This class is similar to a dictionary, (e.g. loaded_data[key]).
    try:
        loaded_data = np.load(file) # lazy loading
    except FileNotFoundError:
        print(f"Error: File not found at path: {file}")
        return {}

    # Calculate the number of meshes from .npz file
    total_arrays = len(loaded_data.files)
    num_meshes = total_arrays // 3  # again, assume each mesh has 3 arrays: vertices, faces, colors
    mol_ids = range(1, num_meshes + 1)
    
    # Output {1: mesh_0, 2: mesh_1, ...}
    meshes_dict = {}

    for i in tqdm(mol_ids, desc="Reconstructing meshes", total=num_meshes, colour='#7BC8F6'):
        key_prefix = f'mesh_{i:04d}'
        try:
            loaded_vertices = loaded_data[f'{key_prefix}_vertices']
            loaded_faces = loaded_data[f'{key_prefix}_faces']
            loaded_colors = loaded_data[f'{key_prefix}_colors']
        except KeyError as e:
            print(f"Error: Missing mesh_{i:04d}. Skipping this molecule.")
            continue
            
        # 3. Reconstruct the trimesh object
        reconstructed_mesh = trimesh.Trimesh(
            vertices=loaded_vertices,
            faces=loaded_faces,
            face_colors=loaded_colors # Note: face_colors is used for coloring faces
        )
        
        # 4. Store the mesh in the dictionary using a 1-based ID
        mesh_id = i
        meshes_dict[mesh_id] = reconstructed_mesh

    return meshes_dict

# Run the function to get the dictionary of meshes
all_reconstructed_meshes = import_meshes('npt-HK4_meshes.npz')

Reconstructing meshes: 100%|██████████| 1501/1501 [00:17<00:00, 87.96it/s]


In [5]:
all_reconstructed_meshes[15].show()